<div dir="rtl">
<h1>آزمونی که نشت آینده را می‌گیرد</h1>
<p>درس 40 از 76 · علّیت را با آزمایش ثابت کنیم · <code dir="ltr">34-causal-test</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html">📖 بازگشت به همین درس</a></p>
<p>یک آزمون رفتاری برای تغییر پیشوند بسازید که اطلاعات را از تصادف Dropout جدا کند.</p><p>پیش‌نیاز: Mask علّی، eval و تفاوت آن با no_grad را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>دو ورودی تا موقعیت دوم یکسان‌اند. برای خروجی‌های همین پیشوند، چه مقایسه‌ای باید در یک مدل علّی برقرار باشد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.attention import CausalSelfAttention
from mini_gpt.config import ModelConfig
attention = CausalSelfAttention(ModelConfig(8,6,4,1,1,0.)).eval()
# A controlled fixture: uniform attention and identity Value/output.
with torch.no_grad():
    attention.qkv.weight.zero_(); attention.qkv.bias.zero_()
    attention.qkv.weight[8:12].copy_(torch.eye(4))
    attention.output.weight.copy_(torch.eye(4)); attention.output.bias.zero_()
x = torch.arange(16.).reshape(1,4,4)
changed = x.clone(); changed[:,2:] += 10
print("same prefix:",torch.equal(x[:,:2],changed[:,:2]))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع prefix_change(Attention, a, b, prefix, causal) بیشینهٔ اختلاف مطلق خروجی دو ورودی را فقط در :prefix برگرداند؛ از همان شیء، حالت eval و no_grad استفاده کنید. خروجی یک float باشد.</p>
</div>

In [ ]:
def prefix_change(attention, a, b, prefix, causal):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = prefix_change(attention,x,changed,2,True)
    if result is None: return False
    assert result < 1e-7
    assert abs(prefix_change(attention,x,changed,2,False)-5.) < 1e-6
    assert prefix_change(attention,x,x,4,False) == 0.
    other = x.clone(); other[:,3:] += 8
    assert prefix_change(attention,x,other,3,True) < 1e-7
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط پرچم causal را تغییر دهید. این نمونهٔ کنترل‌شده عمداً Q/K صفر دارد؛ بدون Mask اثر آینده دقیقاً قابل محاسبه است و به یک Seed خوش‌شانس تکیه نمی‌کند.</p>
</div>

In [ ]:
with torch.no_grad():
    for causal in (False,True):
        difference = attention(x,causal=causal)[:,:2]-attention(changed,causal=causal)[:,:2]
        print(causal,difference.abs().max().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>تابع خراب کل دنباله را مقایسه می‌کند؛ تغییر خروجیِ موقعیت‌های تغییرکرده نشت آینده نیست. تابع prefix_difference(a,b,prefix) را برای دو Tensor خروجی اصلاح کنید.</p>
</div>

In [ ]:
a = torch.zeros(1,4,2)
b = a.clone(); b[:,2:] = 7.
print('wrong whole-sequence test:',(a-b).abs().max().item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def prefix_difference(a, b, prefix):
    # TODO
    return None

In [ ]:
def test_repair():
    result = prefix_difference(a,b,2)
    if result is None: return False
    assert result == 0.
    assert prefix_difference(a,b,3) == 7.
    c = b.clone(); c[:,0,0] = -2.
    assert prefix_difference(a,c,1) == 2.
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>آزمون test_causal_prefix_invariance در tests/test_model.py همین قرارداد را روی MiniGPT کامل می‌سنجد. اینجا از Attention واقعی با وزن‌های کنترل‌شده شروع کردیم؛ مثلثی‌بودن تصویر به‌تنهایی کافی نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر دو اجرای یک ورودی در حالت train متفاوت باشند، قبل از نسبت‌دادن اختلاف به نشت آینده چه عامل دیگری را باید حذف کنید؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/34-causal-test.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>